# Serve vLLM on Google Colab and connect it to the Document Research Assistant

## Why this notebook exists

The FastAPI application is the **assistant product**: it accepts requests, performs RAG and tool calling, and returns validated JSON. An LLM needs a GPU to generate tokens efficiently. This notebook temporarily turns a Colab GPU into that **model server**.

`vLLM` loads an open-source model and serves an OpenAI-compatible `/v1` API. `ngrok` makes Colab's private port reachable from the local FastAPI application. The path is:

```text
Local FastAPI app -> HTTPS ngrok URL -> Colab vLLM server -> Qwen model on Colab GPU
```



In [1]:

!nvidia-smi

Fri Sep  4 07:36:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# First run can take several minutes. pyngrok creates the temporary public tunnel.
!pip -q install -U vllm pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.5/314.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 125.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

In [3]:
import torch

TORCH_VERSION = torch.__version__.split("+")[0]
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)

# Define the torchaudio version explicitly based on available wheels for cu130
# From the error, available versions for cu130 are up to 2.11.0
# We need to specify the full version string including the build tag as listed by pip.
TORCHAUDIO_INSTALL_VERSION = "2.11.0+cu130"

!pip -q uninstall -y torchaudio
!pip -q install --no-cache-dir --force-reinstall --no-deps torchaudio=={TORCHAUDIO_INSTALL_VERSION} --index-url https://download.pytorch.org/whl/cu130

import torchaudio
print("TorchAudio:", torchaudio.__version__, "| compatible import: OK")

PyTorch: 2.13.0+cu130 | CUDA: 13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 44.9 MB/s eta 0:00:00
TorchAudio: 2.11.0+cu130 | compatible import: OK


In [4]:
# Secrets are requested interactively; do NOT paste them into a notebook cell.
import os
from getpass import getpass

NGROK_AUTHTOKEN = getpass('ngrok authtoken: ')
VLLM_API_KEY = getpass('Choose a long API key for vLLM: ')
os.environ['NGROK_AUTHTOKEN'] = NGROK_AUTHTOKEN
os.environ['VLLM_API_KEY'] = VLLM_API_KEY

ngrok authtoken: ··········
Choose a long API key for vLLM: ··········


In [5]:
# A small public model is intentional: prove the deployment first, then scale up if GPU VRAM permits.
import subprocess

MODEL = 'Qwen/Qwen3-0.6B'
SERVED_MODEL_NAME = 'colab-qwen'
log_file = open('/content/vllm.log', 'w')
vllm_process = subprocess.Popen(
    [
        'vllm', 'serve', MODEL,
        '--host', '0.0.0.0',
        '--port', '8000',
        '--served-model-name', SERVED_MODEL_NAME,
        '--api-key', VLLM_API_KEY,
        '--gpu-memory-utilization', '0.80',
        '--max-model-len', '4096',
        '--enable-auto-tool-choice',
        '--tool-call-parser', 'hermes',
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
print(f'vLLM started with PID {vllm_process.pid}. Loading weights can take several minutes.')

vLLM started with PID 2551. Loading weights can take several minutes.


In [6]:
# The logs show a CUDA mismatch error because vLLM started before the TorchAudio fix.
# We need to restart the vLLM process to pick up the new installation.

import time
import requests
import subprocess

# 1. Terminate the old, broken process if it exists
if 'vllm_process' in globals():
    print("Restarting vLLM to apply TorchAudio fixes...")
    vllm_process.terminate()
    vllm_process.wait()

# 2. Restart vLLM (using the same config from the previous cell)
log_file = open('/content/vllm.log', 'w')
vllm_process = subprocess.Popen(
    [
        'vllm', 'serve', MODEL,
        '--host', '0.0.0.0',
        '--port', '8000',
        '--served-model-name', SERVED_MODEL_NAME,
        '--api-key', VLLM_API_KEY,
        '--gpu-memory-utilization', '0.80',
        '--max-model-len', '4096',
        '--enable-auto-tool-choice',
        '--tool-call-parser', 'hermes',
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

# 3. Wait until vLLM is ready
headers = {'Authorization': f'Bearer {VLLM_API_KEY}'}
for attempt in range(90):
    try:
        response = requests.get('http://127.0.0.1:8000/v1/models', headers=headers, timeout=5)
        response.raise_for_status()
        print('vLLM is ready:', response.json())
        break
    except requests.RequestException:
        if attempt % 5 == 0:
            print(f"Waiting for vLLM... (attempt {attempt})")
        time.sleep(5)
else:
    !tail -n 80 /content/vllm.log
    raise RuntimeError('vLLM did not become ready. Read the log above.')

Restarting vLLM to apply TorchAudio fixes...
Waiting for vLLM... (attempt 0)
Waiting for vLLM... (attempt 5)
Waiting for vLLM... (attempt 10)
Waiting for vLLM... (attempt 15)
Waiting for vLLM... (attempt 20)
Waiting for vLLM... (attempt 25)
Waiting for vLLM... (attempt 30)
vLLM is ready: {'object': 'list', 'data': [{'id': 'colab-qwen', 'object': 'model', 'created': 1788507799, 'owned_by': 'vllm', 'root': 'Qwen/Qwen3-0.6B', 'parent': None, 'max_model_len': 4096, 'permission': [{'id': 'modelperm-ac9b1e5a719e3dcd', 'object': 'model_permission', 'created': 1788507799, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [7]:
# Create the HTTPS tunnel only after vLLM is healthy.
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(8000, 'http')
PUBLIC_URL = tunnel.public_url
print('Public vLLM URL:', PUBLIC_URL)
print('Use this base URL in your local .env:', f'{PUBLIC_URL}/v1')

Public vLLM URL: https://repeal-serving-sandstorm.ngrok-free.dev
Use this base URL in your local .env: https://repeal-serving-sandstorm.ngrok-free.dev/v1


In [8]:
# Prove the public endpoint and API-key protection work before connecting your FastAPI app.
payload = {
    'model': SERVED_MODEL_NAME,
    'messages': [{'role': 'user', 'content': 'Reply with exactly: vLLM is ready'}],
    'temperature': 0,
}
response = requests.post(
    f'{PUBLIC_URL}/v1/chat/completions',
    headers={**headers, 'Content-Type': 'application/json'},
    json=payload,
    timeout=120,
)
response.raise_for_status()
print(response.json()['choices'][0]['message']['content'])

<think>
Okay, the user wants me to reply with exactly "vLLM is ready". Let me check the context. The previous message was about vLLM being ready. So I need to make sure I'm not adding anything else. The user might be testing if I can replicate the exact response they want. I should confirm that vLLM is indeed ready and stick to the required format. No extra information or explanations are needed here. Just the exact phrase.
</think>

vLLM is ready.


## Connect your local FastAPI application

On your own computer, put the printed public URL and the same vLLM key in this project's `.env` file:

```env
LLM_PROVIDER=vllm
LLM_MODEL=colab-qwen
VLLM_BASE_URL=https://YOUR-NGROK-URL.ngrok-free.app/v1
VLLM_API_KEY=the-same-key-entered-above
```

Then start your local app with `docker compose up --build` and call `/api/chat`. Keep this Colab notebook running while testing. When you finish, stop the runtime or run the next cell to close the public tunnel.

In [9]:
# # Optional cleanup: run when the demo is finished.
# ngrok.disconnect(PUBLIC_URL)
# vllm_process.terminate()
# print('Tunnel closed and vLLM termination requested.')